In [1]:
import tensorflow as tf
import keras
import numpy as np
import os
import cv2
from sklearn.model_selection import train_test_split
from keras.callbacks import EarlyStopping


2025-10-19 14:35:37.535395: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760898937.563497 3693064 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760898937.572216 3693064 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760898937.594498 3693064 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760898937.594525 3693064 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760898937.594527 3693064 computation_placer.cc:177] computation placer alr

In [ ]:
train_image_folder = '/raid/mpsych/OMAMA/DATA/data/train'
train_npz_folder = '/raid/mpsych/OMAMA/DATA/data/2d_resized_512/images'

In [3]:
# Get lists of image and npz files with labels
# PNG files are synthetic, NPZ files are original
png_files = [(os.path.join(train_image_folder, f), 'png') for f in os.listdir(train_image_folder) if f.endswith('.png')]
npz_files = [(os.path.join(train_npz_folder, f), 'npz') for f in os.listdir(train_npz_folder) if f.endswith('.npz')]

# Combine all files into one list
all_files = png_files + npz_files

print(f"Total files: {len(all_files)}")
print(f"PNG files (synthetic): {len(png_files)}")
print(f"NPZ files (original): {len(npz_files)}")


Total files: 263567
PNG files (synthetic): 99999
NPZ files (original): 163568


In [4]:
print(f"Total files: {len(all_files)}")
print(f"PNG files: {len(png_files)}")
print(f"NPZ files: {len(npz_files)}")

Total files: 263567
PNG files: 99999
NPZ files: 163568


In [5]:
len(npz_files)

163568

In [6]:
# Limit files for faster training
all_files = all_files[:50000]
print(f"Using {len(all_files)} files total")

Using 50000 files total


In [7]:
# Split dataset into train, validation, and test sets
train_files, test_files = train_test_split(all_files, test_size=0.3, random_state=42)
val_files, test_files = train_test_split(test_files, test_size=0.5, random_state=42)

print(f"Train files: {len(train_files)}")
print(f"Validation files: {len(val_files)}")
print(f"Test files: {len(test_files)}")

Train files: 35000
Validation files: 7500
Test files: 7500


In [ ]:
# Image dimensions and batch size
img_height = 512
img_width = 512
batch_size = 32

In [9]:
# Normalization functions (as professor requested)
def normalize_png(image):
    # PNG images: uint8 (0-255) -> normalize to 0-1
    image = image.astype(np.float32)
    image = image / 255.0
    return image

def normalize_npz(npz_data):
    # NPZ data: uint16 (HU values) -> apply W/L then normalize to 0-1
    npz_data = npz_data.astype(np.float32)
    
    # Apply window/level for medical images
    window_center = 400
    window_width = 1200
    window_min = window_center - window_width / 2
    window_max = window_center + window_width / 2
    
    # Clip and normalize
    npz_data = np.clip(npz_data, window_min, window_max)
    npz_data = (npz_data - window_min) / (window_max - window_min)
    
    return npz_data

In [ ]:
def custom_data_generator(file_list, batch_size, img_height, img_width):
    total_files = len(file_list)
    indices = np.arange(total_files)
    np.random.shuffle(indices)

    while True:
        for i in range(0, total_files, batch_size):
            batch_indices = indices[i:i + batch_size]
            batch_images = []
            batch_labels = []

            for idx in batch_indices:
                file_path, file_type = file_list[idx]

                if file_type == 'png':
                    # Load and process PNG file (synthetic)
                    image = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
                    image = cv2.resize(image, (img_width, img_height))
                    image = normalize_png(image)  # Normalize to 0-1
                    label = [1, 0]  # Synthetic
                    
                elif file_type == 'npz':
                    # Load and process NPZ file (original)
                    with np.load(file_path, allow_pickle=True) as data:
                        image = data['data']
                    image = cv2.resize(image, (img_width, img_height))
                    image = normalize_npz(image)  # Apply W/L and normalize to 0-1
                    label = [0, 1]  # Original

                # Add channel dimension
                image = np.expand_dims(image, axis=-1)
                batch_images.append(image)
                batch_labels.append(label)

            # Convert to numpy arrays
            batch_images = np.array(batch_images)
            batch_labels = np.array(batch_labels)
            
            yield (batch_images, batch_labels)




In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3, min_delta=0.001, mode='min')

In [ ]:
NUMBER_OF_CLASSES = 2

In [13]:
# Create data generators
train_generator = custom_data_generator(train_files, batch_size, img_height, img_width)
val_generator = custom_data_generator(val_files, batch_size, img_height, img_width)
test_generator = custom_data_generator(test_files, batch_size, img_height, img_width)


In [14]:
model = keras.models.Sequential()
model.add(keras.layers.Conv2D(32, kernel_size=(3, 3),
                             activation='relu',
                             input_shape=(img_height, img_width, 1)))
model.add(keras.layers.Conv2D(64, (3, 3), activation='relu'))
model.add(keras.layers.MaxPooling2D(pool_size=(2, 2)))
model.add(keras.layers.Dropout(0.25))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(128, activation='relu'))
model.add(keras.layers.Dropout(0.5))
model.add(keras.layers.Dense(NUMBER_OF_CLASSES, activation='softmax'))

/home/a.kanamarlapudi001/miniconda3/envs/tf_gpu_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1760898945.301619 3693064 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 859 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:47:00.0, compute capability: 8.0
2025-10-19 14:35:56.687598: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.97GiB (rounded to 2114060288)requested by op StatelessRandomUniformV2
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary f

ResourceExhaustedError: {{function_node __wrapped__StatelessRandomUniformV2_device_/job:localhost/replica:0/task:0/device:GPU:0}} OOM when allocating tensor with shape[4129024,128] and type float on /job:localhost/replica:0/task:0/device:GPU:0 by allocator GPU_0_bfc [Op:StatelessRandomUniformV2] name: 

In [ ]:
model.compile(loss=keras.losses.categorical_crossentropy,
              optimizer="adadelta",
              metrics=['accuracy'])

In [ ]:
# Model training
try:
    history = model.fit(
        train_generator,
        steps_per_epoch=len(train_files) // batch_size,
        epochs=3,
        validation_data=val_generator,
        validation_steps=len(val_files) // batch_size,
        verbose=1,
        callbacks=[early_stopping]
    )
except Exception as e:
    print("An error occurred during training:", str(e))

# Model evaluation on the test set
test_loss, test_accuracy = model.evaluate(test_generator, steps=len(test_files) // batch_size)
print(f"Test Loss: {test_loss}, Test Accuracy: {test_accuracy}")

In [ ]:
# 3 epochs
test_file = '/raid/mpsych/OMAMA/DATA/data/train/sample_40069.png'
test_image = cv2.imread(test_png_file, cv2.IMREAD_GRAYSCALE)
test_image = cv2.resize(test_image, (img_width, img_height))
test_image = np.expand_dims(test_image, axis=-1)
test_image = test_image / 255.0
test_image = np.expand_dims(test_image, axis=0)  # Add batch dimension

predictions = model.predict(test_image)
print("Predictions:", predictions)
predicted_class = np.argmax(predictions)
print("Predicted Class:", predicted_class)

In [ ]:
# Load and process a synthetic (PNG) test image
test_png_file = '/raid/mpsych/OMAMA/DATA/data/train/sample_10446.png'
test_image = cv2.imread(test_file, cv2.IMREAD_GRAYSCALE)
test_image = cv2.resize(test_image, (img_width, img_height))
test_image = np.expand_dims(test_image, axis=-1)
test_image = test_image / 255.0
test_image = np.expand_dims(test_image, axis=0)  # Add batch dimension

predictions = model.predict(test_image)
print("Predictions:", predictions)
predicted_class = np.argmax(predictions)
print("Predicted Class:", predicted_class)

In [ ]:
test_npz_file = '/raid/mpsych/OMAMA/DATA/data/2d_resized_512/images/100220136299296817993264225430810813957.npz'

with np.load(test_npz_file, allow_pickle=True) as data:
    test_npz = data['data']

# Preprocess the NPZ data using the same normalization as training
test_npz = cv2.resize(test_npz, (img_width, img_height))
test_npz = normalize_npz(test_npz)  # Use the same W/L normalization as training
test_npz = np.expand_dims(test_npz, axis=-1)  # Add channel dimension
test_npz = np.expand_dims(test_npz, axis=0)  # Add batch dimension

# Make prediction
prediction = model.predict(test_npz)
print("Predictions:", prediction)
predicted_class = np.argmax(prediction)
class_label = 'Real' if predicted_class == 0 else 'Synthetic'
print(f"Predicted Class: {class_label}, Probability: {np.max(prediction)}")


In [ ]:
# 3 epochs
# test_file = '/raid/mpsych/OMAMA/DATA/data/train/sample_40069.png'
# test_image = cv2.imread(test_file, cv2.IMREAD_GRAYSCALE)
# test_image = cv2.resize(test_image, (img_width, img_height))
# test_image = np.expand_dims(test_image, axis=-1)
# test_image = test_image / 255.0
# test_image = np.expand_dims(test_image, axis=0)  # Add batch dimension

# predictions = model.predict(test_image)
# print("Predictions:", predictions)
# predicted_class = np.argmax(predictions)
# print("Predicted Class:", predicted_class)